In [ ]:
import dataclasses
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    _WAVELET_BAND_FREQ_RESOLUTION_HZ,
    _wavelet_transform,
    analyzers_to_datasets,
    load_analyzers,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Inter-Subject Frequency-Channel ICA / PCA on Wavelet Power

**Approach 2** — concatenate *person × time* into observations,
keep *channel × frequency* as features.

```
(n_subjects, n_channels, n_freqs, n_times)
  → reshape →  (n_subjects × n_times,  n_channels × n_freqs)
```

PCA / ICA find a small set of **spatial-spectral components** — recurring
channel × frequency patterns shared across subjects and time.

**Analyses:**

1. PCA scree plot (variance explained)
2. Component channel × frequency maps
3. Per-subject activation time courses
4. Component topographic maps (channel marginal)
5. Inter-subject similarity of component activations
6. Band-resolved component loadings
7. Temporal dynamics of components (group mean activation)
8. Component correlation matrix (PCA vs ICA)


## Configuration


In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / _WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True

# ── Subsets ────────────────────────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5
N_CHANNELS_SUBSET: int | None = 32
N_TIMES_SUBSET: int | None = 10000

# ── Decomposition settings ────────────────────────────────────────────────────
N_COMPONENTS_PCA = 20  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42

# ── Storage directory ─────────────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "plots" / "intersubject"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading


In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.


In [ ]:
broadband_datasets = _wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.


In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects \u00d7 channels \u00d7 freqs \u00d7 times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

## Inter-Subject Reshape

Flatten `(n_channels, n_freqs)` into one feature axis and
`(n_subjects, n_times)` into the observation axis.

```
X_inter:  (n_subjects × n_times,  n_channels × n_freqs)
```

Each row is one (subject, time-point) observation of the full
spatial-spectral power vector.  PCA/ICA will discover recurring
patterns in this space.


In [ ]:
# Reshape 4-D → 2-D: (n_subjects * n_times,  n_channels * n_freqs)
# First transpose to (n_subjects, n_times, n_channels, n_freqs), then flatten
bb_transposed = bb_data.transpose(0, 3, 1, 2)  # (S, T, C, F)
X_inter = bb_transposed.reshape(n_subjects * n_times, n_channels * n_freqs)
print(f"Inter-subject matrix shape: {X_inter.shape}")
print(f"  Rows = {n_subjects} subjects \u00d7 {n_times} time points")
print(f"  Cols = {n_channels} channels \u00d7 {n_freqs} frequencies")

# Z-score each column (feature) across observations
col_mean = X_inter.mean(axis=0, keepdims=True)
col_std = X_inter.std(axis=0, keepdims=True) + 1e-10
X_inter_z = (X_inter - col_mean) / col_std
print("Z-scored inter-subject matrix ready.")

---
## 1 — PCA Scree Plot (Variance Explained)

The intrinsic dimensionality of the channel × frequency space
tells us how many independent spatial-spectral modes exist in the
wavelet power data.


In [ ]:
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca.fit(X_inter_z)  # (n_obs, n_features)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot \u2014 {LABEL}")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance \u2014 {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

---
## 2 — Component Channel × Frequency Maps

Each PCA component is a vector of length `n_channels × n_freqs`.
Reshape to `(n_channels, n_freqs)` and display as a heatmap — this
is the **spatial-spectral fingerprint** of the component.


In [ ]:
# PCA components: (K, n_channels * n_freqs)
components = pca.components_  # (K, C*F)

n_show = min(6, N_COMPONENTS_PCA)
_vlim = np.percentile(
    np.abs(components[:n_show].reshape(n_show, n_channels, n_freqs)), 99
)
fig, axes = plt.subplots(2, (n_show + 1) // 2, figsize=(5 * ((n_show + 1) // 2), 8))
axes = axes.ravel()

for i in range(n_show):
    comp_map = components[i].reshape(n_channels, n_freqs)  # (C, F)
    im = axes[i].imshow(
        comp_map,
        aspect="auto",
        origin="lower",
        extent=[FREQS[0], FREQS[-1], 0, n_channels],
        cmap="RdBu_r",
        vmin=-_vlim,
        vmax=_vlim,
    )
    axes[i].set_xlabel("Frequency (Hz)")
    axes[i].set_ylabel("Channel index")
    axes[i].set_title(f"PC {i + 1}  ({explained[i] * 100:.1f}%)", fontsize=10)
    plt.colorbar(im, ax=axes[i], shrink=0.8)

# Hide unused axes
for j in range(n_show, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"Spatial-Spectral Component Maps \u2014 {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_channel_freq_maps.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3 — Per-Subject Activation Time Courses

Project each (subject, time-point) observation onto the PCA axes.
The resulting scores matrix has shape `(n_subjects × n_times, K)`.
Reshape to `(n_subjects, n_times, K)` and plot the per-subject
activation of each component over time.


In [ ]:
# PCA scores: (n_subjects * n_times, K)
scores = pca.transform(X_inter_z)

# Reshape to (n_subjects, n_times, K)
scores_3d = scores.reshape(n_subjects, n_times, N_COMPONENTS_PCA)

n_show = min(4, N_COMPONENTS_PCA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

_ylim = np.percentile(np.abs(scores_3d[:, :, :n_show]), 99)
colors = sns.color_palette("husl", n_subjects)
for i, ax in enumerate(axes):
    for s in range(n_subjects):
        ax.plot(
            time,
            scores_3d[s, :, i],
            lw=0.5,
            alpha=0.6,
            color=colors[s],
            label=f"S{s + 1}" if i == 0 else None,
        )
    ax.set_ylim(-_ylim, _ylim)
    ax.set_ylabel(f"PC {i + 1}")
    ax.set_title(
        f"Component {i + 1} activation  ({explained[i] * 100:.1f}%)", fontsize=10
    )

axes[0].legend(fontsize=7, ncol=n_subjects, loc="upper right")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Per-Subject Component Activations \u2014 {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_subject_activations.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 4 — Component Topographic Maps (Channel Marginal)

Marginalize the component vectors over frequency to get a per-channel
weight.  Plot as scalp topographies using `mne.viz.plot_topomap`.


In [ ]:
import mne  # noqa: E402
from mne.viz import plot_topomap  # noqa: E402

# Channel marginal: mean loading over frequencies (signed, for diverging colormap)
components_2d = components.reshape(N_COMPONENTS_PCA, n_channels, n_freqs)
channel_marginal = components_2d.mean(axis=2)  # (K, n_channels)

# Get MNE Info
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_PCA)
_vlim = np.percentile(np.abs(channel_marginal[:n_show]), 99)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
for i, ax in enumerate(axes):
    im, _ = plot_topomap(
        channel_marginal[i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        # vmin=-_vlim, vmax=_vlim,
    )
    ax.set_title(f"PC {i + 1}", fontsize=10)

fig.suptitle(f"Component Topomaps (freq-averaged) \u2014 {LABEL}", fontsize=12)
plt.colorbar(im, ax=axes[-1], label="mean loading")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_topomaps.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5 — Inter-Subject Similarity of Component Activations

For each component, compute the Pearson correlation of the activation
time course between every pair of subjects.  High inter-subject
correlation means the spatial-spectral mode is stimulus-driven rather
than idiosyncratic.


In [ ]:
n_show = min(6, N_COMPONENTS_PCA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)

for i, ax in enumerate(axes):
    # scores_3d[:, :, i] has shape (n_subjects, n_times)
    corr_mat = np.corrcoef(scores_3d[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"PC {i + 1}", fontsize=10)

fig.suptitle(f"Inter-Subject Correlation per Component \u2014 {LABEL}", fontsize=12)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "intersubject_similarity.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6 — Band-Resolved Component Loadings

For each component, aggregate loadings within each canonical
frequency band (delta, theta, alpha, beta, gamma) to show which
bands dominate the spatial-spectral pattern.


In [ ]:
# components_2d: (K, n_channels, n_freqs)
band_loadings = {}  # band -> (K,)
for band, (lo, hi) in FREQUENCY_BANDS.items():
    band_mask = (FREQS >= lo) & (FREQS <= hi)
    # Mean |loading| over channels and band frequencies
    band_loadings[band] = np.abs(components_2d[:, :, band_mask]).mean(axis=(1, 2))

bands_list = list(FREQUENCY_BANDS.keys())
n_show = min(6, N_COMPONENTS_PCA)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_show)
width = 0.15
for j, band in enumerate(bands_list):
    ax.bar(
        x + j * width,
        band_loadings[band][:n_show],
        width,
        label=band,
    )

ax.set_xticks(x + width * (len(bands_list) - 1) / 2)
ax.set_xticklabels([f"PC {i + 1}" for i in range(n_show)])
ax.set_ylabel("Mean |loading|")
ax.set_title(f"Band-Resolved Component Loadings \u2014 {LABEL}")
ax.legend(fontsize=8)

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "band_resolved_loadings.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7 — Temporal Dynamics of Components (Group Mean)

Average the per-subject activation of each component across subjects
to get a **group-mean** temporal profile.  Overlay ±1 SD to show
inter-individual variability.


In [ ]:
n_show = min(4, N_COMPONENTS_PCA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

_ylim = np.percentile(np.abs(scores_3d[:, :, :n_show]), 99)
for i, ax in enumerate(axes):
    group_mean = scores_3d[:, :, i].mean(axis=0)  # (n_times,)
    group_std = scores_3d[:, :, i].std(axis=0)

    ax.plot(time, group_mean, lw=1, color="steelblue", label="Mean")
    ax.set_ylim(-_ylim, _ylim)
    ax.fill_between(
        time,
        group_mean - group_std,
        group_mean + group_std,
        alpha=0.25,
        color="steelblue",
        label="\u00b11 SD",
    )
    ax.set_ylabel(f"PC {i + 1}")
    ax.set_title(
        f"Component {i + 1} group mean  ({explained[i] * 100:.1f}%)", fontsize=10
    )
    ax.legend(fontsize=7, loc="upper right")

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Group-Mean Component Activations \u2014 {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "group_mean_activations.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8 — Component Correlation Matrix (PCA vs ICA)

Fit FastICA on the same data and compare the correlation structure
between PCA (orthogonal by construction) and ICA (approximately
independent) components.


In [ ]:
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_scores = ica.fit_transform(X_inter_z)  # (n_obs, K_ica)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: PCA
pca_corr = np.corrcoef(scores[:, :N_COMPONENTS_PCA].T)
im0 = axes[0].imshow(pca_corr, vmin=-1, vmax=1, cmap="RdBu_r")
axes[0].set_title("PCA component correlations")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Component")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Panel B: ICA
ica_corr = np.corrcoef(ica_scores.T)
im1 = axes[1].imshow(ica_corr, vmin=-1, vmax=1, cmap="RdBu_r")
axes[1].set_title("ICA component correlations")
axes[1].set_xlabel("Component")
axes[1].set_ylabel("Component")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

fig.suptitle(f"Cross-Component Correlations \u2014 {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "cross_component_correlation.png", dpi=150, bbox_inches="tight"
    )
plt.show()

---
## 8.1 — ICA Spatial-Spectral Component Maps

Reshape ICA mixing matrix columns into `(n_channels, n_freqs)` maps
for visual comparison with the PCA components above.


In [ ]:
# ICA mixing matrix: (n_features, n_components) = (C*F, K_ica)
ica_mixing = ica.mixing_  # (C*F, K_ica)

n_show = min(6, N_COMPONENTS_ICA)
_vlim_ica = np.percentile(np.abs(ica_mixing[:, :n_show]), 99)
fig, axes = plt.subplots(2, (n_show + 1) // 2, figsize=(5 * ((n_show + 1) // 2), 8))
axes = axes.ravel()

for i in range(n_show):
    comp_map = ica_mixing[:, i].reshape(n_channels, n_freqs)
    im = axes[i].imshow(
        comp_map,
        aspect="auto",
        origin="lower",
        extent=[FREQS[0], FREQS[-1], 0, n_channels],
        cmap="RdBu_r",
        vmin=-_vlim_ica,
        vmax=_vlim_ica,
    )
    axes[i].set_xlabel("Frequency (Hz)")
    axes[i].set_ylabel("Channel index")
    axes[i].set_title(f"IC {i + 1}", fontsize=10)
    plt.colorbar(im, ax=axes[i], shrink=0.8)

for j in range(n_show, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"ICA Spatial-Spectral Maps \u2014 {LABEL}", fontsize=13, y=1.01)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_channel_freq_maps.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8.2 — ICA Topographic Maps — Mean Loading Across Subjects

The ICA mixing matrix `(C×F, K_ica)` gives the spatial-spectral
pattern for each component.  Reshape to `(C, F, K)` and average
over frequencies to get the channel loading per component.

To capture per-subject variation, we weight these channel loadings
by each subject's mean activation (from the ICA scores).  The
**mean** across subjects gives the average spatial pattern.


In [ ]:
# ICA mixing: (C*F, K_ica) → reshape to (C, F, K)
ica_mixing_3d = ica_mixing.reshape(n_channels, n_freqs, N_COMPONENTS_ICA)

# Channel loading (freq-marginalized): (C, K)
ica_ch_loading = ica_mixing_3d.mean(axis=1)  # (C, K)

# Reshape ICA scores to (S, T, K)
ica_scores_3d = ica_scores.reshape(n_subjects, n_times, N_COMPONENTS_ICA)

# Per-subject mean activation: (S, K)
subject_mean_act = ica_scores_3d.mean(axis=1)  # (S, K)

# Per-subject channel loading: outer product → (S, C, K)
per_subj_ch = (
    ica_ch_loading[np.newaxis, :, :]  # (1, C, K)
    * subject_mean_act[:, np.newaxis, :]  # (S, 1, K)
)  # (S, C, K)

# Mean across subjects → (C, K)
ica_ch_mean = per_subj_ch.mean(axis=0)

n_show = min(6, N_COMPONENTS_ICA)
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    im, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        # vmin=-_vlim_mean,
        # vmax=_vlim_mean,
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"ICA Topomaps — Mean Loading Across Subjects — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="mean loading")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_topomap_mean.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()

---
## 8.3 — ICA Topographic Maps — Variance Across Subjects

Compute the **variance** of the subject-weighted channel loading
across subjects.  High variance at a channel means the component
is active with very different strength for different individuals.


In [ ]:
# Variance across subjects → (C, K)
ica_ch_var = per_subj_ch.var(axis=0)  # (C, K)

n_show = min(6, N_COMPONENTS_ICA)
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    im, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=ax,
        show=False,
        cmap="YlOrRd",
        # vmin=0,
        # vmax=_vmax_var,
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"ICA Topomaps — Variance Across Subjects — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="variance of loading")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_topomap_variance.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()

---
## 8.4 — Inter-Individual IC Correlations

For each ICA component, compute the Pearson correlation between every
pair of subjects using their IC activation time courses.
High correlations indicate the IC captures a temporal pattern that is
shared across individuals — evidence of stimulus-locked processing.


In [ ]:
# ica_scores_3d: (S, T, K) — per-subject IC activation time courses
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each subject's time course for IC i: (n_times,)
    corr_mat = np.corrcoef(ica_scores_3d[:, :, i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Inter-Individual IC Correlations (time courses) — {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "ica_interindividual_correlation.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()

---
## Summary

Available variables for further analysis:

| Variable | Shape | Description |
|----------|-------|-------------|
| `X_inter_z` | `(S×T, C×F)` | Z-scored inter-subject observation matrix |
| `pca` | — | Fitted PCA object |
| `scores_3d` | `(S, T, K)` | Per-subject PCA component activations |
| `components_2d` | `(K, C, F)` | PCA components as channel × freq maps |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(S×T, K_ica)` | ICA component activations |
| `ica_mixing` | `(C×F, K_ica)` | ICA mixing matrix |

**Analyses implemented:**

1. PCA scree plot (variance explained)
2. Spatial-spectral component maps (channel × frequency)
3. Per-subject activation time courses
4. Component topographic maps (channel marginal)
5. Inter-subject similarity of component activations
6. Band-resolved component loadings
7. Group-mean temporal dynamics of components
8. Cross-component correlation (PCA vs ICA)

See `README.md` in this directory for the full analysis rationale,
alternative decomposition strategies, and ideas for future extensions.
